# ComplaintIQ - sentence embeddings in supervised modeling (`09_supervised_embeddings`)

`04b` established that raw TF-IDF on the narrative subset reached **PR-AUC ~0.28** with
**top-10% lift ~4.5x**. The question: can sentence embeddings (a pretrained transformer) replace
or augment TF-IDF to raise either or both of those numbers?

This notebook **compares three text representations** on the `monetary_relief` task (same chronological
split, stratified sample, and imbalance-aware metrics as `04b`):
1. **TF-IDF baseline** (rebuilt from `04b` as a reference).
2. **Sentence embeddings** alone.
3. **Stacked features**: embeddings + TF-IDF, feeding both to the model.

All three are reported against the **narrative subset's own test base rate** (like `04b`), so the
lift numbers are comparable.

## How to read this notebook
Spark reads and splits the data chronologically (80-20 split via epoch-days quantile); stratified
sampling draws 300k from each split (preserving class balance). The text path filters for complaints
with narratives, vectorizes them (TF-IDF or embeddings), and fits logistic regression with balanced
class weights.

**Focus on:** PR-AUC and top-10% lift — did embeddings beat TF-IDF on either?

> **Go deeper:**
> - [sentence-transformers: all-MiniLM-L6-v2](https://www.sbert.net/docs/pretrained_models.html): *small, fast model for production use, ~8 min.*
> - [LIME / integrated gradients](https://github.com/marcotcr/lime): *understanding which features each model weighs, ~10 min.*

## Setup

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from __future__ import annotations
from pyspark.sql import DataFrame as SparkDataFrame
from typing import Any
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
from complaintiq.metrics import report, top_k_lift
from complaintiq.sampling import stratified_pandas
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
import time

sns.set_theme(style="whitegrid", palette="colorblind", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("sklearn loaded")

In [ ]:
# GPU detection - prove we're running on GPU serverless
try:
    import torch

    cuda_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU"
    print(f"\nGPU Detection:")
    print(f"  CUDA available: {cuda_available}")
    print(f"  Device: {device_name}")
except Exception as e:
    cuda_available = False
    device_name = f"Error: {e}"
    print(f"GPU Detection failed: {device_name}")

---
## 1. Load + chronological split

Same foundation as `04b` — load complaints with date_received, compute epoch-days, and split at the
80th percentile (train older, test recent). Keep narrative fields for the text path.

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR if VOLUME_DIR.exists() else Path("..") / "data"
path = data_dir / "complaints.parquet"
if not path.exists():
    raise FileNotFoundError("data/complaints.parquet not found. Run `make parquet` first.")

META_COLS = ["product", "sub_product", "issue", "sub_issue", "submitted_via", "state", "tags"]
spark_df = (
    spark.read.parquet(str(path))
    .select(
        "monetary_relief",
        "has_narrative",
        "complaint_text",
        F.to_date("date_received").alias("date_received"),
        *META_COLS,
    )
    .dropna(subset=["date_received"])
)

spark_df = spark_df.withColumn("epoch", F.datediff("date_received", F.lit("1970-01-01")))
cut = spark_df.approxQuantile("epoch", [0.80], 0.001)[0]
train_sdf = spark_df.filter(F.col("epoch") <= cut).drop("epoch")
test_sdf = spark_df.filter(F.col("epoch") > cut).drop("epoch")
print(f"train rows: {train_sdf.count():,}  |  test rows: {test_sdf.count():,}")

---
## 2. Stratified sample + helpers

Same stratified sampling as `04b` to preserve class balance in pandas. Report function captures
PR-AUC, ROC-AUC, Brier score, and lift at top-1%/5%/10%.

In [ ]:
train = stratified_pandas(train_sdf, 300_000)
test = stratified_pandas(test_sdf, 300_000)
print(f"train sample: {len(train):,} (pos {train['monetary_relief'].mean():.4%})")
print(f"test sample:  {len(test):,} (pos {test['monetary_relief'].mean():.4%})")
results = []

---
## 3. Text baseline (TF-IDF + logistic regression)

Rebuild `04b`'s TF-IDF baseline on the narrative subset. This is our reference for whether
embeddings improve.

In [ ]:
train_txt = train[train["has_narrative"] & train["complaint_text"].notna()]
test_txt = test[test["has_narrative"] & test["complaint_text"].notna()]
y_train_narrative = train_txt["monetary_relief"].to_numpy()
y_test_narrative = test_txt["monetary_relief"].to_numpy()

print(
    f"narrative subset — train {len(train_txt):,}  test {len(test_txt):,}  "
    f"(subset base rate {y_test_narrative.mean():.4%})"
)

tfidf = TfidfVectorizer(max_features=20_000, min_df=5, stop_words="english")
tfidf_train = tfidf.fit_transform(train_txt["complaint_text"])
tfidf_test = tfidf.transform(test_txt["complaint_text"])

text_lr = LogisticRegression(max_iter=200, class_weight="balanced")
text_lr.fit(tfidf_train, y_train_narrative)
scores_text = text_lr.predict_proba(tfidf_test)[:, 1]
results.append(report("logreg_tfidf_text", y_test_narrative, scores_text))

> **TF-IDF baseline:** reference to beat. If embeddings PR-AUC is higher, or lift at top-10% is
> higher, embeddings win that metric.

---
## 4. Sentence embeddings representation

Encode narratives with `all-MiniLM-L6-v2` (small, fast pretrained transformer). Feed the dense
vectors directly to logistic regression.

> **GPU note:** sentence-transformers auto-uses CUDA on serverless GPU infrastructure; this runs
> fast on A10. On CPU, expect 5-10min encode time.

In [ ]:
from sentence_transformers import SentenceTransformer

t0 = time.time()
model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode train narratives -> dense vectors, normalized for cosine similarity.
embedding_train = model.encode(
    train_txt["complaint_text"].tolist(),
    batch_size=256,
    show_progress_bar=False,
    normalize_embeddings=True,
)
embedding_test = model.encode(
    test_txt["complaint_text"].tolist(),
    batch_size=256,
    show_progress_bar=False,
    normalize_embeddings=True,
)
encode_time = time.time() - t0
print(
    f"embeddings shape: train {embedding_train.shape}  test {embedding_test.shape}  ({encode_time:.1f}s)"
)

# Logistic regression on embeddings.
emb_lr = LogisticRegression(max_iter=200, class_weight="balanced")
emb_lr.fit(embedding_train, y_train_narrative)
scores_emb = emb_lr.predict_proba(embedding_test)[:, 1]
results.append(report("logreg_embeddings_only", y_test_narrative, scores_emb))

> **Sentence embeddings solo:** does this beat TF-IDF? Compare PR-AUC and lift.

---
## 5. Stacked features (embeddings + TF-IDF)

Feed both representations to the model — each embedding dimension + each TF-IDF dimension.
This tests whether embeddings and TF-IDF capture complementary signal.

In [ ]:
from scipy.sparse import hstack

# Stack embeddings (dense) + TF-IDF (sparse). Need to convert embeddings to scipy sparse for stacking.
from scipy.sparse import csr_matrix

embedding_train_sparse = csr_matrix(embedding_train)
embedding_test_sparse = csr_matrix(embedding_test)

X_stacked_train = hstack([tfidf_train, embedding_train_sparse])
X_stacked_test = hstack([tfidf_test, embedding_test_sparse])
print(f"stacked features: train {X_stacked_train.shape}  test {X_stacked_test.shape}")

# Logistic regression on the combined feature space (with increased max_iter for the larger model).
stack_lr = LogisticRegression(max_iter=300, class_weight="balanced", solver="lbfgs")
stack_lr.fit(X_stacked_train, y_train_narrative)
scores_stack = stack_lr.predict_proba(X_stacked_test)[:, 1]
results.append(report("logreg_stacked_tfidf_embeddings", y_test_narrative, scores_stack))

> **Stacked features:** if this outperforms both embeddings and TF-IDF alone, both representations
> carry unique signal; if it only matches the best solo model, one dominates.

---
## 6. Comparison scoreboard

In [ ]:
board = pd.DataFrame(results).set_index("model")
print("Results DataFrame:")
print(board.round(4))

# Metrics export BEFORE plotting (so we capture data even if plots fail)
import json as _json

metrics = {
    "notebook": "09_supervised_embeddings",
    "gpu": {"cuda_available": bool(cuda_available), "device_name": str(device_name)},
    "split": {"train_size": int(len(train)), "test_size": int(len(test))},
    "narrative_subset": {
        "train_size": int(len(train_txt)),
        "test_size": int(len(test_txt)),
        "base_rate": float(y_test_narrative.mean()),
    },
    "embedding_encode_time_sec": float(encode_time),
    "results": results,
    "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
try:
    dbutils.fs.put(
        "/Volumes/workspace/complaintiq/data/metrics_09.json",
        _json.dumps(metrics, indent=2),
        overwrite=True,
    )
    print("[OK] wrote metrics_09.json")
    print(f"  GPU info: {metrics['gpu']}")
except Exception as e:
    print(f"[FAIL] Error writing metrics: {e}")

# Plot PR-AUC comparison (use board DataFrame, not results list)
try:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    sns.barplot(
        data=board.reset_index(),
        x="pr_auc",
        y="model",
        hue="model",
        palette="colorblind",
        legend=False,
        ax=axes[0],
    )
    axes[0].set_title("PR-AUC comparison")
    axes[0].set_xlabel("PR-AUC")

    # Plot top-10% lift
    sns.barplot(
        data=board.reset_index(),
        x="top10pct_lift",
        y="model",
        hue="model",
        palette="colorblind",
        legend=False,
        ax=axes[1],
    )
    axes[1].set_title("Top-10% lift comparison")
    axes[1].set_xlabel("Lift over base rate")

    # Plot top-5% lift
    sns.barplot(
        data=board.reset_index(),
        x="top5pct_lift",
        y="model",
        hue="model",
        palette="colorblind",
        legend=False,
        ax=axes[2],
    )
    axes[2].set_title("Top-5% lift comparison")
    axes[2].set_xlabel("Lift over base rate")

    plt.tight_layout()
    plt.show()
    print("[OK] plots completed successfully")
except Exception as e:
    print(f"[FAIL] Plotting error (metrics already saved): {type(e).__name__}: {e}")

In [ ]:
import json as _json

metrics = {
    "notebook": "09_supervised_embeddings",
    "split": {"train_size": int(len(train)), "test_size": int(len(test))},
    "narrative_subset": {
        "train_size": int(len(train_txt)),
        "test_size": int(len(test_txt)),
        "base_rate": float(y_test_narrative.mean()),
    },
    "embedding_encode_time_sec": float(encode_time),
    "results": results,
    "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
dbutils.fs.put(
    "/Volumes/workspace/complaintiq/data/metrics_09.json",
    _json.dumps(metrics, indent=2),
    overwrite=True,
)
print("wrote metrics_09.json")

---
## 8. Takeaways

> - **If embeddings PR-AUC > TF-IDF:** consider it the new text baseline for monetary_relief.
> - **If embeddings lift < TF-IDF:** text has different levers; investigate what each model learns
>   (local explanation tools like LIME).
> - **If stacked > best solo:** future feature engineering should combine both, not pick one.
> - **Production path:** embeddings are faster to train than tuned TF-IDF pipelines and more robust
>   to paraphrasing; if PR-AUC is in the ballpark, embeddings may be the better deploy choice.
>
> **Measured result (GPU, A10G):** on the narrative subset, TF-IDF PR-AUC **0.271** beats
> embeddings-alone **0.241**; stacking both is best at **0.281** (top-1% lift 6.1x). So for the
> *supervised* relief signal, word-matching TF-IDF wins and embeddings help only as a complement -
> the opposite of the *unsupervised* clustering result (notebook 10), where embeddings win. The best
> representation is **task-dependent**; see `docs/FINDINGS.md` (finding 6). *(narrative test subset
> ~5,600 rows, so treat lift as indicative.)*
>
> Next: full-corpus clustering at scale with embeddings (notebook 10).